In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
print("Token cargado:", bool(os.environ.get("HF_TOKEN")))

Token cargado: True


In [2]:
from huggingface_hub import whoami
print(whoami()["name"])

grisselblanco


In [3]:
import pandas as pd

ofertas = pd.DataFrame([
    ("Coto",      "Leche entera La Serenísima sachet 1L",      1200),
    ("Carrefour", "Leche La Serenisima Entera 1 litro",        1150),
    ("Día",       "Leche entera Sancor 1L",                    1100),
    ("Coto",      "Leche descremada La Serenísima 1L",         1250),
    ("Día",       "Aceite de girasol Cocinero 900ml",          2400),
    ("Carrefour", "Aceite Cocinero girasol botella 900 cc",    2300),
    ("Coto",      "Aceite de girasol Natura 1.5L",             3600),
    ("Día",       "Arroz largo fino Gallo 1kg",                 1500),
    ("Carrefour", "Arroz Gallo largo fino paquete 1 kg",       1450),
    ("Coto",      "Detergente Magistral limón 750ml",           2100),
    ("Carrefour", "Lavavajillas Magistral limon 750 ml",        2000),
], columns=["super", "producto", "precio"])

ofertas

,super,producto,precio
0,Coto,Leche entera La Serenísima sachet 1L,1200
1,Carrefour,Leche La Serenisima Entera 1 litro,1150
2,Día,Leche entera Sancor 1L,1100
3,Coto,Leche descremada La Serenísima 1L,1250
4,Día,Aceite de girasol Cocinero 900ml,2400
5,Carrefour,Aceite Cocinero girasol botella 900 cc,2300
6,Coto,Aceite de girasol Natura 1.5L,3600
7,Día,Arroz largo fino Gallo 1kg,1500
8,Carrefour,Arroz Gallo largo fino paquete 1 kg,1450
9,Coto,Detergente Magistral limón 750ml,2100


In [4]:
casos = {
    "leche entera serenisima 1 litro": {0, 1},
    "aceite girasol cocinero 900": {4, 5},
    "arroz gallo 1kg": {7, 8},
    "detergente magistral limon": {9, 10},
}

casos

{'leche entera serenisima 1 litro': {0, 1},
 'aceite girasol cocinero 900': {4, 5},
 'arroz gallo 1kg': {7, 8},
 'detergente magistral limon': {9, 10}}

In [5]:
import numpy as np
from transformers import pipeline

def cargar_pipe(repo):
    return pipeline("feature-extraction", model=repo, device=-1)

def embeber(pipe, textos, prefijo=""):
    entradas = [prefijo + texto for texto in textos]
    salidas = pipe(entradas, truncation=True, max_length=128)

    vectores = []
    for salida in salidas:
        tokens = np.asarray(salida, dtype=np.float32)

        # Algunos formatos devuelven una dimensión extra para el lote.
        if tokens.ndim == 3:
            tokens = tokens[0]

        if tokens.ndim != 2:
            raise ValueError(f"Forma inesperada de salida: {tokens.shape}")

        vectores.append(tokens.mean(axis=0))

    vectores = np.stack(vectores)
    normas = np.linalg.norm(vectores, axis=1, keepdims=True)

    if np.any(normas == 0):
        raise ValueError("El modelo produjo un vector de norma cero.")

    return vectores / normas

def rankear(pipe, consulta, textos, prefijo=""):
    vector_consulta = embeber(pipe, [consulta], prefijo)[0]
    vectores_productos = embeber(pipe, textos, prefijo)
    return vectores_productos @ vector_consulta

modelos = {
    "paraphrase-MiniLM": (
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        ""
    ),
    "e5-small": (
        "intfloat/multilingual-e5-small",
        "query: "
    ),
}

modelos

c:\Users\grise\Proyectos\Proyecto_Expensa_House\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'paraphrase-MiniLM': ('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
  ''),
 'e5-small': ('intfloat/multilingual-e5-small', 'query: ')}

In [6]:
import time
import numpy as np
import pandas as pd

textos = ofertas["producto"].astype(str).tolist()
resultados = []

for nombre, (repo, prefijo) in modelos.items():
    print(f"Cargando y probando: {nombre}")
    inicio = time.time()

    pipe = cargar_pipe(repo)
    aciertos = 0

    for consulta, correctos in casos.items():
        scores = rankear(pipe, consulta, textos, prefijo)
        top2 = set(np.argsort(scores)[::-1][:2])
        aciertos += len(top2 & correctos) / len(correctos)

    resultados.append({
        "modelo": nombre,
        "precision_top2": round(aciertos / len(casos), 2),
        "segundos": round(time.time() - inicio, 1),
    })

tabla_resultados = pd.DataFrame(resultados)
tabla_resultados

Cargando y probando: paraphrase-MiniLM


c:\Users\grise\Proyectos\Proyecto_Expensa_House\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\grise\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\grise\Proyectos\Proyecto_Expensa_House\.venv\L

Cargando y probando: e5-small


c:\Users\grise\Proyectos\Proyecto_Expensa_House\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\grise\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1223.00it/s]


,modelo,precision_top2,segundos
0,paraphrase-MiniLM,0.88,104.2
1,e5-small,1.00,86.9


In [7]:
from huggingface_hub import InferenceClient
import time
import numpy as np
import pandas as pd

client = InferenceClient(provider="hf-inference")
resultados_api = []

for nombre, (repo, prefijo) in modelos.items():
    print(f"Probando por API: {nombre}")
    inicio = time.time()
    aciertos = 0

    # E5 fue entrenado para usar el prefijo "query:" en similitud semántica.
    textos_api = [prefijo + texto for texto in textos]

    for consulta, correctos in casos.items():
        consulta_api = prefijo + consulta

        scores = client.sentence_similarity(
            sentence=consulta_api,
            other_sentences=textos_api,
            model=repo,
        )

        top2 = set(np.argsort(scores)[::-1][:2])
        aciertos += len(top2 & correctos) / len(correctos)

    resultados_api.append({
        "modelo": nombre,
        "precision_top2": round(aciertos / len(casos), 2),
        "segundos": round(time.time() - inicio, 1),
        "tipo_de_salida": type(scores).__name__,
    })

tabla_api = pd.DataFrame(resultados_api)
tabla_api

Probando por API: paraphrase-MiniLM


HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/hf-inference/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/pipeline/sentence-similarity' (Request ID: Root=1-6acd5e53-2e749d8a00e3edf97d2c94f8;aabcc896-1e4d-4d31-ac3f-614a7dc6a430)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.